# Worksheet 4.2: Models and the Climate


## Part A: Connecting to the Reading

You read Piers Forster's article about Manabe's 1967 paper. Discuss the following questions with your group (we'll just stick with the Mini-Project groups for today).

**Exercise A.1.** Forster writes that Manabe "needed to make the physics as simple as possible" because of computing limitations. But he also says this simplicity "became a strength." In your own words, why might simplicity be an advantage rather than just a limitation?

*Your answer:*



**Exercise A.2.** According to the article, what was the key problem that "earlier attempts to estimate the warming from carbon dioxide increases had floundered" on? What did Manabe figure out that others hadn't?

*Your answer:*



---

## Setup

Run this cell to load our climate models. You don't need to understand the code—just click the play button.

In [ ]:
#@title Load the Climate Model (click the play button to run)
#@markdown This loads our simplified climate models.

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import matplotlib.patches as patches

# Set up nicer plots
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

# ============================================================
# THE CLIMATE MODEL (simplified energy balance)
# ============================================================

# Pre-industrial CO2 level (baseline)
CO2_PREINDUSTRIAL = 280  # ppm
CO2_BASELINE = 300  # Manabe's baseline

# Earth's baseline temperature
TEMP_BASELINE = 15.0  # degrees C (approximate global average)

def calculate_radiative_forcing(co2_ppm):
    """
    Calculate the radiative forcing from CO2 change.
    Standard formula used by IPCC.
    """
    if co2_ppm <= 0:
        return -999  # Invalid
    return 5.35 * np.log(co2_ppm / CO2_BASELINE)

def calculate_temperature(co2_ppm, water_vapor_feedback=True):
    """
    Calculate Earth's surface temperature based on CO2 level.

    With water vapor feedback: warming evaporates more water,
    water vapor is a greenhouse gas, so warming amplifies.

    Without feedback: just the direct CO2 effect.
    """
    forcing = calculate_radiative_forcing(co2_ppm)

    # Climate sensitivity parameter (degrees C per W/m squared)
    if water_vapor_feedback:
        # With feedbacks: ~0.8 degrees C per W/m squared (gives ~3 degrees C for doubling)
        sensitivity = 0.8
    else:
        # Without feedbacks: ~0.3 degrees C per W/m squared (gives ~1.2 degrees C for doubling)
        sensitivity = 0.31

    temp_change = forcing * sensitivity
    return TEMP_BASELINE + temp_change

def get_manabe_result(co2_ppm):
    """
    What Manabe & Wetherald (1967) actually found.
    """
    if co2_ppm == 150:
        return -2.28  # Cooling from halving CO2
    elif co2_ppm == 600:
        return 2.36   # Warming from doubling CO2
    else:
        # Interpolate using log relationship
        return 3.4 * np.log(co2_ppm / 300) / np.log(2)

# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def draw_thermometer(ax, temperature, label="", show_baseline=True):
    """
    Draw a thermometer visualization.
    """
    ax.clear()

    # Thermometer dimensions
    bulb_center = (0.5, 0.15)
    bulb_radius = 0.08
    tube_width = 0.06
    tube_bottom = 0.2
    tube_top = 0.9

    # Temperature range for display
    temp_min = 10
    temp_max = 25

    # Calculate fill height
    temp_fraction = (temperature - temp_min) / (temp_max - temp_min)
    temp_fraction = np.clip(temp_fraction, 0, 1)
    fill_height = tube_bottom + temp_fraction * (tube_top - tube_bottom)

    # Color based on temperature
    if temperature < 14:
        color = '#3498db'  # Blue for cold
    elif temperature < 16:
        color = '#e74c3c'  # Red for normal
    else:
        color = '#c0392b'  # Dark red for hot

    # Draw thermometer outline
    # Bulb
    circle = plt.Circle(bulb_center, bulb_radius, fill=False,
                        edgecolor='black', linewidth=2)
    ax.add_patch(circle)

    # Tube outline
    tube = patches.FancyBboxPatch(
        (0.5 - tube_width/2, tube_bottom), tube_width, tube_top - tube_bottom,
        boxstyle="round,pad=0.01", fill=False,
        edgecolor='black', linewidth=2
    )
    ax.add_patch(tube)

    # Fill bulb
    circle_fill = plt.Circle(bulb_center, bulb_radius * 0.85,
                             facecolor=color, edgecolor='none')
    ax.add_patch(circle_fill)

    # Fill tube
    tube_fill = patches.Rectangle(
        (0.5 - tube_width/2 + 0.005, tube_bottom),
        tube_width - 0.01, fill_height - tube_bottom,
        facecolor=color, edgecolor='none'
    )
    ax.add_patch(tube_fill)

    # Draw tick marks and labels
    for temp in range(10, 26, 5):
        y = tube_bottom + (temp - temp_min) / (temp_max - temp_min) * (tube_top - tube_bottom)
        ax.plot([0.5 + tube_width/2, 0.5 + tube_width/2 + 0.03], [y, y], 'k-', linewidth=1)
        ax.text(0.5 + tube_width/2 + 0.05, y, f'{temp} C', va='center', fontsize=10)

    # Baseline marker
    if show_baseline:
        baseline_y = tube_bottom + (TEMP_BASELINE - temp_min) / (temp_max - temp_min) * (tube_top - tube_bottom)
        ax.plot([0.5 - tube_width/2 - 0.08, 0.5 - tube_width/2], [baseline_y, baseline_y],
                'g--', linewidth=2)
        ax.text(0.5 - tube_width/2 - 0.1, baseline_y, 'Baseline\n(15 C)',
                ha='right', va='center', fontsize=9, color='green')

    # Temperature display
    ax.text(0.5, 0.02, f'{temperature:.2f} C', ha='center', fontsize=16, fontweight='bold')

    # Label
    if label:
        ax.set_title(label, fontsize=14, fontweight='bold')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')

def draw_earth_diagram(ax, co2_level, temperature, with_feedback=False):
    """
    Draw a simple Earth energy balance diagram.
    """
    ax.clear()

    # Earth
    earth = plt.Circle((0.5, 0.3), 0.15, facecolor='#3498db', edgecolor='black', linewidth=2)
    ax.add_patch(earth)

    # Atmosphere (thickness based on CO2)
    atm_thickness = 0.05 + 0.1 * (co2_level / 600)
    atmosphere = plt.Circle((0.5, 0.3), 0.15 + atm_thickness,
                           facecolor='none', edgecolor='orange',
                           linewidth=3, linestyle='--', alpha=0.7)
    ax.add_patch(atmosphere)

    # Add water vapor layer if feedback is on
    if with_feedback:
        water_layer = plt.Circle((0.5, 0.3), 0.15 + atm_thickness + 0.03,
                               facecolor='none', edgecolor='#5dade2',
                               linewidth=2, linestyle=':', alpha=0.7)
        ax.add_patch(water_layer)

    # Sun rays coming in
    for x_offset in [-0.1, 0, 0.1]:
        ax.annotate('', xy=(0.5 + x_offset, 0.55), xytext=(0.5 + x_offset, 0.85),
                   arrowprops=dict(arrowstyle='->', color='#f4d03f', lw=3))

    # Heat trapped (more for higher CO2)
    n_arrows = int(2 + 3 * (co2_level / 600))
    if with_feedback:
        n_arrows = int(n_arrows * 1.5)  # More arrows for feedback
    for i in range(n_arrows):
        angle = np.pi/4 + i * np.pi / (n_arrows + 1)
        x = 0.5 + 0.22 * np.cos(angle)
        y = 0.3 + 0.22 * np.sin(angle)
        dx = -0.08 * np.cos(angle)
        dy = -0.08 * np.sin(angle)
        ax.annotate('', xy=(x + dx, y + dy), xytext=(x, y),
                   arrowprops=dict(arrowstyle='->', color='red', lw=2, alpha=0.6))

    # Labels
    ax.text(0.5, 0.95, 'Sunlight', ha='center', fontsize=12)
    ax.text(0.88, 0.5, f'CO2 blanket\n({co2_level} ppm)', ha='center', fontsize=10, color='orange')
    if with_feedback:
        ax.text(0.12, 0.5, 'Water vapor\n(feedback)', ha='center', fontsize=9, color='#5dade2')
    ax.text(0.5, 0.3, f'{temperature:.1f} C', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    title = "Earth's Energy Balance"
    if with_feedback:
        title += " (with water vapor feedback)"
    ax.set_title(title, fontsize=12)

print("Climate model loaded successfully!")
print("")
print("You now have access to simplified climate models.")
print("Let's see what they can teach us about our planet...")

---

## Part B: Your Prediction

Before we run any experiments, let's see what you think will happen.

**The Question:** If we **double** the amount of CO2 in the atmosphere (from 300 ppm to 600 ppm), how much will Earth's average temperature increase?

Discuss with your group, and use the slider below to record your prediction:

In [ ]:
#@title Make Your Prediction
#@markdown Press "play" and use the slider to record your prediction.

prediction_slider = widgets.FloatSlider(
    value=1.0,
    min=0,
    max=10,
    step=0.1,
    description='My prediction:',
    style={'description_width': 'initial'},
    readout_format='.1f'
)

prediction_label = widgets.HTML(
    value='<h3 style="color: #3498db;">Temperature will increase by: 1.0 C</h3>'
)

def update_prediction_label(change):
    val = change['new']
    prediction_label.value = f'<h3 style="color: #3498db;">Temperature will increase by: {val:.1f} C</h3>'

prediction_slider.observe(update_prediction_label, names='value')

print("If CO2 doubles from 300 ppm to 600 ppm, I predict Earth will warm by:")
print("")
display(prediction_slider)
display(prediction_label)
print("")
print("(Remember your prediction -- we'll compare it to what the models show!)")

---

## Part C: A Simple Energy Balance Model (Pre-Manabe)

Before Manabe's breakthrough, climate scientists used a basic approach: CO2 traps heat, so more CO2 means a warmer Earth. Simple.

This model treats the atmosphere like a single blanket. It captures the direct effect of CO2 but ignores what happens *after* the initial warming occurs.

**Run the cell below to experiment with this simple model:**

In [ ]:
#@title Simple Energy Balance Model (Pre-Manabe)
#@markdown Press "play". This model shows only the direct effect of CO2, without any feedbacks.

# Create output area for the plot
simple_output = widgets.Output()

# CO2 slider
simple_co2_slider = widgets.IntSlider(
    value=300,
    min=150,
    max=800,
    step=10,
    description='CO2 (ppm):',
    style={'description_width': 'initial'},
    continuous_update=True
)

# Quick buttons
simple_btn_300 = widgets.Button(description='300 ppm (baseline)', button_style='success')
simple_btn_600 = widgets.Button(description='600 ppm (doubled)', button_style='warning')

def set_simple_co2(value):
    simple_co2_slider.value = value

simple_btn_300.on_click(lambda b: set_simple_co2(300))
simple_btn_600.on_click(lambda b: set_simple_co2(600))

def update_simple_experiment(change):
    with simple_output:
        clear_output(wait=True)

        co2 = simple_co2_slider.value
        temp = calculate_temperature(co2, water_vapor_feedback=False)  # NO FEEDBACK
        temp_change = temp - TEMP_BASELINE

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Thermometer
        draw_thermometer(ax1, temp, f"Earth's Temperature at {co2} ppm CO2")

        # Earth diagram
        draw_earth_diagram(ax2, co2, temp, with_feedback=False)

        plt.tight_layout()
        plt.show()

        # Display change from baseline
        if temp_change > 0.01:
            change_text = f"<span style='color: red; font-size: 18px;'>+{temp_change:.2f} C from baseline</span>"
        elif temp_change < -0.01:
            change_text = f"<span style='color: blue; font-size: 18px;'>{temp_change:.2f} C from baseline</span>"
        else:
            change_text = f"<span style='color: green; font-size: 18px;'>At baseline (15 C)</span>"

        display(HTML(f"<center>{change_text}</center>"))
        display(HTML("<center><i>Simple model: CO2 effect only</i></center>"))

simple_co2_slider.observe(update_simple_experiment, names='value')

# Create layout
simple_buttons = widgets.HBox([simple_btn_300, simple_btn_600])
simple_ui = widgets.VBox([
    widgets.HTML("<h4>Quick select:</h4>"),
    simple_buttons,
    widgets.HTML("<h4>Or use the slider:</h4>"),
    simple_co2_slider,
    simple_output
])

display(simple_ui)

# Initial display
update_simple_experiment(None)

---

### Part C Exercises

**Exercise C.1.** Set CO2 to 300 ppm (the baseline), then to 600 ppm (doubled). How much warming does this simple model predict for doubled CO2?

*Your answer:*



---

## Part D: Adding Manabe's Key Insight

Let's see what happens when we add water vapor feedback to our model.

**Run the cell below to compare the two models side by side:**

In [ ]:
#@title Compare: Simple Model vs. Manabe's Model (with Water Vapor Feedback)
#@markdown Press "play" to see the comparison

# Create output
feedback_output = widgets.Output()

# CO2 slider for this experiment
co2_slider_2 = widgets.IntSlider(
    value=600,
    min=150,
    max=800,
    step=10,
    description='CO2 (ppm):',
    style={'description_width': 'initial'}
)

def update_feedback_experiment(change):
    with feedback_output:
        clear_output(wait=True)

        co2 = co2_slider_2.value
        feedback_on = True

        # Calculate both scenarios
        temp_with_feedback = calculate_temperature(co2, water_vapor_feedback=True)
        temp_without_feedback = calculate_temperature(co2, water_vapor_feedback=False)

        current_temp = temp_with_feedback if feedback_on else temp_without_feedback

        fig, axes = plt.subplots(1, 3, figsize=(14, 5))

        # Thermometer WITHOUT feedback
        draw_thermometer(axes[0], temp_without_feedback, "Without Feedback\n(Simple Model)")
        if not feedback_on:
            axes[0].add_patch(patches.Rectangle((0, 0), 1, 1, fill=False,
                                                 edgecolor='blue', linewidth=4))

        # Thermometer WITH feedback
        draw_thermometer(axes[1], temp_with_feedback, "With Water Vapor Feedback\n(Manabe's Model)")
        if feedback_on:
            axes[1].add_patch(patches.Rectangle((0, 0), 1, 1, fill=False,
                                                 edgecolor='blue', linewidth=4))

        # Comparison bar chart
        axes[2].clear()
        change_without = temp_without_feedback - TEMP_BASELINE
        change_with = temp_with_feedback - TEMP_BASELINE

        bars = axes[2].bar(['Without\nFeedback', 'With\nFeedback'],
                          [change_without, change_with],
                          color=['#95a5a6', '#e74c3c'])

        axes[2].set_ylabel('Warming (C)', fontsize=11)
        axes[2].set_title(f'Warming at {co2} ppm CO2', fontsize=12, fontweight='bold')
        axes[2].axhline(y=0, color='black', linewidth=0.5)

        # Add value labels
        for bar, val in zip(bars, [change_without, change_with]):
            if val >= 0:
                axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.1,
                            f'+{val:.2f} C', ha='center', fontsize=12, fontweight='bold')
            else:
                axes[2].text(bar.get_x() + bar.get_width()/2, val - 0.2,
                            f'{val:.2f} C', ha='center', fontsize=12, fontweight='bold')

        # Calculate amplification
        if abs(change_without) > 0.01:
            amplification = change_with / change_without
            extra_warming = change_with - change_without
            percent_increase = (extra_warming / change_without) * 100 if change_without > 0 else 0
            axes[2].text(0.5, -0.18, f'Amplification: {amplification:.2f}x ({percent_increase:.0f}% increase)',
                        transform=axes[2].transAxes, ha='center', fontsize=11,
                        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

        plt.tight_layout()
        plt.show()

co2_slider_2.observe(update_feedback_experiment, names='value')

display(widgets.HTML("<h4>CO2 level:</h4>"))
display(co2_slider_2)
display(feedback_output)

# Initial display
update_feedback_experiment(None)

---

### Part D Exercises

Set CO2 to 600 ppm (doubled) and toggle the water vapor feedback ON and OFF.

**Exercise D.1.** How much warming occurs WITHOUT water vapor feedback?

*Your answer:*



**Exercise D.2.** How much warming occurs WITH water vapor feedback?

*Your answer:*



**Exercise D.3.** How close is our model's prediction (with feedback) to Manabe's actual 1967 result of 2.36 degrees C?

*Your answer:*



---

## Part E: Why Does Water Vapor Amplify Warming?

**Exercise E.1.** The reading doesn't fully explain *why* water vapor creates a feedback loop. Based on what you know: if CO2 warms the Earth slightly, why would that lead to even more warming? (Hint: think about what happens to water when things get warmer, and what water vapor does in the atmosphere.)

*Your answer:*



**Exercise E.2.** This is called a "positive feedback loop." In your own words, describe the chain of events: How does a small amount of warming from CO2 lead to even more warming through water vapor?

*Your answer:*



---

## Part F: Past, Present, and Future

Let's use our model (with water vapor feedback) to examine different scenarios:

In [ ]:
#@title Past, Present, and Future Scenarios
#@markdown See how temperature changes across different CO2 levels.

# Key CO2 levels
scenarios = {
    'Pre-industrial\n(1850)': 280,
    'Manabe baseline\n(1967)': 300,
    'Today\n(2024)': 420,
    'Doubled\n(future?)': 560,
    'Worst case\n(2100?)': 800
}

fig, ax = plt.subplots(figsize=(12, 6))

names = list(scenarios.keys())
co2_vals = list(scenarios.values())
temps = [calculate_temperature(co2, water_vapor_feedback=True) for co2 in co2_vals]
temp_changes = [t - TEMP_BASELINE for t in temps]

colors = ['#2ecc71', '#27ae60', '#f39c12', '#e74c3c', '#c0392b']

bars = ax.bar(names, temps, color=colors, edgecolor='black', linewidth=1.5)

# Add baseline line
ax.axhline(y=TEMP_BASELINE, color='green', linestyle='--', linewidth=2, label='Baseline (300 ppm)')

# Add temperature labels
for bar, temp, change, co2 in zip(bars, temps, temp_changes, co2_vals):
    label = f'{temp:.1f} C'
    if abs(change) > 0.1:
        if change > 0:
            label += f'\n(+{change:.1f} C)'
        else:
            label += f'\n({change:.1f} C)'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            label, ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('Global Average Temperature (C)', fontsize=12)
ax.set_title("Earth's Temperature: Past, Present, and Future Scenarios", fontsize=14, fontweight='bold')
ax.set_ylim(13, 22)
ax.legend(loc='upper left')

# Add CO2 labels at bottom
for i, (name, co2) in enumerate(scenarios.items()):
    ax.text(i, 13.3, f'{co2} ppm', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.show()

print("")
print("Note: These are simplified projections from our model.")
print("Real climate involves many more factors, but the basic physics remains the same.")

---

### Part F Exercises

**Exercise F.1.** According to our model, how much has Earth warmed since pre-industrial times (280 ppm) to today (420 ppm)?

*Your answer:*



**Exercise F.2.** If CO2 reaches 800 ppm by 2100, what temperature does the model predict? How much total warming is that from pre-industrial levels?

*Your answer:*



---

## Part H: En-ROADS Climate Simulator

Now let's explore a more sophisticated model used by policymakers around the world. En-ROADS was developed by MIT and Climate Interactive and is used in workshops with world leaders, business executives, and policymakers.

**Access the Model:** Go to https://en-roads.climateinteractive.org/scenario.html

With your group, spend some time exploring the interface. Move the sliders and watch how they affect the temperature projection for 2100.

---

### Part H Exercises

**Exercise H.1: Extreme Scenario (High)**

What combination of slider values gives you the **highest** temperature change by 2100? Describe the world represented by these extreme values, copy and paste the "Share Scenario" link to your exact scenario, and paste a screenshot of your model.

*Your answer:*



**Exercise H.2: Extreme Scenario (Low)**

What combination of slider values gives you the **lowest** temperature change by 2100? What are the most impactful changes in this scenario? Again, copy and paste the "Share Scenario" link to your exact scenario, and paste a screenshot of your model

*Your answer:*



**Exercise H.4: Model Assumptions**

What are some assumptions built into this model? How might these assumptions influence the results?

*Your answer:*



**Exercise H.5: Model Limitations**

What are some limitations of this model? What aspects of social behavior or climate change might it not capture fully?

*Your answer:*



---

## Summary

Today you:

1. **Connected the reading to hands-on exploration** -- You tested the ideas from Forster's article yourself

2. **Compared simple vs. feedback models** -- You saw why water vapor amplification matters so much

3. **Explored a policy-relevant model** -- You used En-ROADS to see how models inform real-world decisions

---

### Key Takeaways for the Quant Unit:

- **Models are simplified versions of reality** -- They leave things out on purpose
- **Simple can be powerful** -- Manabe's genius was knowing what to include and what to ignore
- **Models help us see what we can't observe directly** -- We can't experiment on the real Earth, but we can experiment on models
- **Feedback loops matter** -- Small effects can amplify into big ones
- **Models involve assumptions** -- Understanding those assumptions is crucial for interpreting results

---

---
Worksheet created by Ethan C. Brown and Claude (see [conversation 1](https://claude.ai/share/aa3a359c-ebb1-48b1-9f95-e40bc88c10d4) [conversation 2](https://claude.ai/share/70d95b52-404a-4b52-8a3e-41f631c17959)).